# COS801 RQ7.1 — Model compression benchmark
This notebook runs the same leakage-audited split through FP32 baselines, INT8 quantisation, structured pruning, and knowledge distillation. RQ7.1 full mode targets the original RGB/color PlantVillage task with all 38 crop-disease classes. Start with smoke mode; only full-mode results belong in the report.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')

def run(cmd, cwd=None):
    print('$', ' '.join(str(x) for x in cmd))
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed

Mounted at /content/drive


In [ ]:
# Use the project files already stored in Google Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive')
REPO_ROOT = DRIVE_ROOT / 'COS801_Project'
PROJECT_ROOT = REPO_ROOT / 'rq7.1-model-compression'
# Drive keeps the durable dataset copy; /content is used for faster runtime reads.
DRIVE_DATA_DIR = DRIVE_ROOT / 'COS801_data' / 'rq71_plantvillage'
LOCAL_DATA_DIR = Path('/content/rq71_plantvillage')
SMOKE_DATA_DIR = DRIVE_ROOT / 'COS801_data' / 'rq71_smoke'
OUTPUT_ROOT = DRIVE_ROOT / 'COS801_results' / 'rq7.1'
# False = real 38-class PlantVillage experiment. Change to True only for a fast synthetic pipeline check.
SMOKE_MODE = False

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found in Drive: {PROJECT_ROOT}')

DATA_DIR = SMOKE_DATA_DIR if SMOKE_MODE else LOCAL_DATA_DIR
OUTPUT_DIR = OUTPUT_ROOT / ('smoke' if SMOKE_MODE else 'full')
CONFIG = PROJECT_ROOT / 'configs' / ('smoke.json' if SMOKE_MODE else 'full.json')

print('PROJECT_ROOT   =', PROJECT_ROOT)
print('DRIVE_DATA_DIR =', DRIVE_DATA_DIR)
print('DATA_DIR       =', DATA_DIR)
print('OUTPUT_DIR     =', OUTPUT_DIR)
print('CONFIG         =', CONFIG)

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime', 'torch-pruning', 'scikit-learn', 'datasets'])
os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

In [ ]:
# Keep the shared configuration reproducible while supplying machine-specific paths.
# Full mode copies data to /content for faster reads; results still save to Drive.
if SMOKE_MODE:
    run([sys.executable, 'make_smoke_dataset.py', '--output', str(DATA_DIR), '--seed', '37'])
else:
    if not DRIVE_DATA_DIR.exists():
        raise FileNotFoundError(f'Drive dataset folder not found: {DRIVE_DATA_DIR}')

    required_splits = ['train', 'val', 'test']
    if not LOCAL_DATA_DIR.exists():
        print('Copying dataset from Drive to fast local Colab disk:', LOCAL_DATA_DIR)
        print('This is still a one-time cost for this runtime, but avoids writing a huge tar archive back to Drive.')
        shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, dirs_exist_ok=True)
    else:
        print('Local dataset folder already exists:', LOCAL_DATA_DIR)

    missing = [split for split in required_splits if not (LOCAL_DATA_DIR / split).exists()]
    if missing:
        raise FileNotFoundError(f'Local dataset copy is incomplete; missing splits: {missing}')
    print('Using fast local dataset:', LOCAL_DATA_DIR)

cfg = json.loads(CONFIG.read_text())
cfg['data_dir'] = str(DATA_DIR)
cfg['output_dir'] = str(OUTPUT_DIR)
RUNTIME_CONFIG = Path('/content/rq71_runtime_config.json')
RUNTIME_CONFIG.write_text(json.dumps(cfg, indent=2))
print('Runtime config written to:', RUNTIME_CONFIG)
cfg

## 1. EDA and leakage gate

In [ ]:
run([sys.executable, 'run_experiment.py', 'eda', '--config', str(RUNTIME_CONFIG)])

## 2. FP32 baselines

In [ ]:
run([sys.executable, 'run_experiment.py', 'train', '--config', str(RUNTIME_CONFIG), '--model', 'mobilenet_v3_small'])
run([sys.executable, 'run_experiment.py', 'train', '--config', str(RUNTIME_CONFIG), '--model', 'efficientnet_b0'])

## 3. Compression experiments
Select the better validation baseline as the teacher/checkpoint before a full run.

In [ ]:
TEACHER = Path(cfg['output_dir']) / 'mobilenet_v3_small' / 'baseline.pt'
run([sys.executable, 'run_experiment.py', 'quantize', '--config', str(RUNTIME_CONFIG), '--checkpoint', str(TEACHER)])
run([sys.executable, 'run_experiment.py', 'prune', '--config', str(RUNTIME_CONFIG), '--checkpoint', str(TEACHER)])
run([sys.executable, 'run_experiment.py', 'distill', '--config', str(RUNTIME_CONFIG), '--checkpoint', str(TEACHER)])

## 4. Results table

In [ ]:
import pandas as pd
results = pd.read_csv(Path(cfg['output_dir']) / 'metrics.csv')
results.sort_values(['accuracy', 'model_size_mb'], ascending=[False, True])